# 7.22 為什麼有些MTP分支串成一條鏈？


平行版本讀到「雨」時，第二條分支直接猜「了」。但「雨停」之後接「了」，和「雨又」之後接「了」，是不同的文字問題。能不能讓較遠預測沿著中間的token前進，而不是每條都只拿同一份雨的特徵？

**DeepSeek-V3的MTP**採用順序輔助模組。主核心先讀已知前文；第一個額外模組再拿到主核心的特徵，以及訓練材料裡真正的下一項，處理後預測再下一項。這個模組是一段額外的神經網路，將兩份資訊合在一起，不是把答案文字直接當成模型輸出。

沿用句子「雨停了」。讀到「雨」的主核心負責預測「停」；訓練時，額外模組取得「雨」的特徵和**標準token「停」的向量**，負責預測「了」。若再接一個模組，就取得前一模組的特徵和標準token「了」，預測〈E〉。下圖示範兩個額外模組，為的是看清資料流。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/p7-7.22-sequential-mtp.svg")))

為什麼訓練時可以提供「停」？因為它教的是「雨停之後接什麼」，目標是下一項「了」。就像先前[對話訓練](https://birdhackor.github.io/tiny-perceptron-vlm/7.4.html)把真正答案前半段放進輸入，以便教後半段。這種提供正確前文的做法叫**teacher forcing，使用標準序列作為訓練前文**；它本身不等於MTP。原本的一步預測也能這樣訓練，MTP額外增加的是同一位置出發的未來任務。

這沒有讓主核心在預測「停」時偷讀「停」：「停」的向量只流向額外模組。也沒有先把「了」交給第一個額外模組再叫它答「了」。每一條箭頭都要對照「目前讀了什麼、接下來預測什麼」，才能分清合理前文和洩漏答案。

實際DeepSeek-V3採一個額外深度：主模型預測下一項，額外模組預測再下一項；不是本圖兩個額外模組的設定。它的分支還共用查字與輸出詞表的權重，所以「不同未來任務」不必等同「每條都有獨立整張詞表矩陣」。平行分支與順序模組都屬MTP，資料流卻不同。

使用時沒有標準未來token可讀。可以只保留主模型，照原本方式生成；也可以讓額外模組接剛猜出的token，提出後續候選，再由主模型核對。後者不是把訓練時的標準答案搬到推論，它還需要[16.14的草稿驗證](https://birdhackor.github.io/tiny-perceptron-vlm/16.14.html)。

練習把訓練句子改成「雨又停」。仍從讀到「雨」的主核心出發：主模型目標是什麼？第一個額外模組收到哪個標準token、目標又是什麼？依序是「又」、收到「又」、預測「停」。若畫成第一個模組先收到「停」，再以「停」當目標，便不是這裡的對齊。

<details>
<summary>選讀：來源與這張圖的簡化範圍</summary>

[DeepSeek-V3 Technical Report第2.2節與圖3](https://arxiv.org/pdf/2412.19437v2)交代順序模組、共享嵌入／輸出層和額外代價；[第4.2節](https://arxiv.org/pdf/2412.19437v2)記錄實際深度D=1。本圖省略合併特徵的投影、正規化與模組內部Transformer，保留訓練前文和答案的關係。

原文支持這種架構與其實驗條件下的結果，不保證本課小模型也改善。架構選擇在[第14章](https://birdhackor.github.io/tiny-perceptron-vlm/chapter-14.html)展開，成品的取捨見[19.8](https://birdhackor.github.io/tiny-perceptron-vlm/19.8.html)。

</details>
